# Workforce Talent Risk

Identify companies with low satisfaction, limited training and large employee populations

In [ ]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

df = pd.read_csv("data/software_companies_dataset_v2.csv")

# Prepare missing values for reliable widgets and Plotly charts.
categorical_columns = [
    "Company_Name", "Industry", "Headquarters_City", "Country",
    "Ownership_Type", "Customer_Segment", "Primary_Cloud", "Risk_Rating"
]
for column in categorical_columns:
    if column in df.columns:
        df[column] = df[column].fillna("Unknown").astype(str).str.strip()

numeric_columns = [
    "Employees", "Annual_Revenue", "Profit_Margin", "Market_Share",
    "R&D_Spending", "Average_Salary", "Training_Hours_Per_Employee",
    "Employee_Satisfaction", "Adoption_Rate_AI", "Adoption_Rate_Cloud",
    "Adoption_Rate_Blockchain"
]
for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")
        df[column] = df[column].fillna(df[column].median())

df["Annual_Revenue"] = df["Annual_Revenue"].clip(lower=1)
df["Employees"] = df["Employees"].clip(lower=1)

threshold = widgets.FloatSlider(value=3.0, min=1, max=5, step=.1, description="Satisfaction")
training = widgets.FloatSlider(value=40, min=0, max=100, step=5, description="Training <")
out = widgets.Output()

def render(*_):
    watch = df[(df["Employee_Satisfaction"] <= threshold.value) &
               (df["Training_Hours_Per_Employee"] <= training.value)].copy()
    with out:
        clear_output(wait=True)
        display(widgets.HTML(f"<h3>Talent-risk companies: {len(watch)}</h3>"))
        px.scatter(watch, x="Training_Hours_Per_Employee", y="Employee_Satisfaction",
                   size="Employees", color="Industry", hover_name="Company_Name",
                   title="Talent-risk matrix").show()
        display(watch[["Company_Name","Country","Industry","Employees",
                       "Employee_Satisfaction","Training_Hours_Per_Employee"]]
                .sort_values("Employees", ascending=False).head(30))

threshold.observe(render, names="value")
training.observe(render, names="value")
display(widgets.HBox([threshold, training]), out)
render()